# E5 — Bootstrap Threshold Sweep

**Tinker RL Project — PES University MTech Capstone (Group 6)**

| Field | Value |
|-------|-------|
| **Hypothesis** | GRPO fails when step-0 pass@1 ≈ 0% (no seed signal); succeeds above ~7% |
| **Model** | `Qwen/Qwen3-8B` |
| **Dataset** | GSM8K (difficulty-stratified by gold solution length) |
| **Bins** | `easy` (Q1, short solutions) vs `hardest` (Q5, long solutions) |
| **Key metric** | `train/frac_zero_signal` — fraction of steps where all G rollouts = 0 |
| **Finding it tests** | Finding 1 from report: minimum capacity / seed signal requirement |

**Run both cells below twice** — once with `DIFFICULTY_BIN=easy`, once with `DIFFICULTY_BIN=hardest`.
Compare `train/frac_zero_signal` and convergence curves in WandB.

In [ ]:
!pip install -q atroposlib==0.3.0 \
    'git+https://github.com/thinking-machines-lab/tinker.git' \
    datasets transformers wandb pydantic python-dotenv math-verify latex2sympy2-extended

In [ ]:
!git clone https://github.com/pes-llm-research/tinker-rl-lab.git
%cd tinker-rl-lab/atropos
!pip install -e . -q

In [ ]:
import os

# Set credentials
os.environ['TINKER_API_KEY'] = 'YOUR_TINKER_API_KEY'
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN'

# *** CHANGE THIS to 'hardest' for the second run ***
DIFFICULTY_BIN = 'easy'   # or 'hardest'
os.environ['BOOTSTRAP_DIFFICULTY_BIN'] = DIFFICULTY_BIN

config = f'configs/bootstrap_threshold_{DIFFICULTY_BIN}.yaml'
os.environ['TINKER_CONFIG_PATH'] = config
print(f'Running bin: {DIFFICULTY_BIN} | config: {config}')

In [ ]:
!nohup python -m atroposlib.server > /tmp/atropos_server.log 2>&1 &
import time; time.sleep(5)
print('Atropos coordinator started')

In [ ]:
import os
config = os.environ['TINKER_CONFIG_PATH']
!nohup python -m tinker_atropos.environments.bootstrap_threshold_tinker \
    --config {config} \
    > /tmp/bootstrap_env.log 2>&1 &
import time; time.sleep(10)
print('Environment started')
!tail -5 /tmp/bootstrap_env.log

In [ ]:
import os
config = os.environ['TINKER_CONFIG_PATH']
!python launch_training.py --config {config}

In [ ]:
# RESULTS — paste data here for BOTH bins after training completes
# Format: steps = [...], rewards = [...], zero_signal = [...]

results = {
    'easy': {
        'steps': [],
        'rewards': [],
        'frac_zero_signal': [],  # from wandb train/frac_zero_signal
    },
    'hardest': {
        'steps': [],
        'rewards': [],
        'frac_zero_signal': [],
    },
}

if results['easy']['steps'] and results['hardest']['steps']:
    import matplotlib.pyplot as plt
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy curves
    ax1.plot(results['easy']['steps'], results['easy']['rewards'], color='#2ecc71', linewidth=2, label='easy (Q1)')
    ax1.plot(results['hardest']['steps'], results['hardest']['rewards'], color='#e74c3c', linewidth=2, label='hardest (Q5)')
    ax1.set_xlabel('Training Step'); ax1.set_ylabel('Accuracy'); ax1.legend(); ax1.grid(True, alpha=0.3)
    ax1.set_title('Accuracy by Difficulty Bin')

    # Zero-signal fraction
    ax2.plot(results['easy']['steps'], results['easy']['frac_zero_signal'], color='#2ecc71', linewidth=2, label='easy')
    ax2.plot(results['hardest']['steps'], results['hardest']['frac_zero_signal'], color='#e74c3c', linewidth=2, label='hardest')
    ax2.set_xlabel('Training Step'); ax2.set_ylabel('Frac Zero-Signal Steps'); ax2.legend(); ax2.grid(True, alpha=0.3)
    ax2.set_title('Fraction of Steps with Zero GRPO Signal')
    ax2.set_ylim(-0.05, 1.05)

    plt.suptitle('E5: Bootstrap Threshold Sweep — Qwen3-8B on GSM8K', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('Run both bins first, then paste results here.')